<div style="background-color:red; padding: 18px; border-radius: 3px; text-align:center; font-size:2.5em; font-weight:bold; color:#222; margin-bottom:25px; letter-spacing:1px;">
Predicting Donor Response for Social Good 
</div>


# <h2 style="border-bottom: 4px solid red; width:fit-content; margin: 0 auto 25px auto; padding-bottom:6px; font-weight:bold;">SHOWCASE OF PRE PROCESSING, DEV VERSION</h2>

_Data Mining II 2025/2026_

Project by:
Francisco Gomes (20221810), Margarida Marchão (20221901), Marta Alves (20221890), Pedro Coimbras (20211573)

 ## <h3 style="border-bottom: 4px solid RED; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">1.  Importing needed libraries and data</h3>

In [17]:
"""Importing core libraries for data handling and visualization"""
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

"""Adding the project root to the Python path so notebook imports work correctly"""
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

"""Importing custom EDA helper functions and visualizations from the utils_eda, and utils_plots modules"""
from utils.utils_eda import *
from utils.utils_plots import (
    plot_invalid_value_counts,
    plot_boxplots_numeric,
    plot_correlation_heatmap,
    plot_missing_percentages,
    plot_missing_overlap_heatmap,
    plot_target_correlation,
    plot_missing_vs_target,
    plot_outlier_boxplots,
    plot_outlier_percentages_iqr,
    plot_outlier_histograms,
    plot_categorical_frequencies,
    plot_categorical_target_rate,
)

In [18]:
"""Loading the training dataset for exploratory data analysis"""
data = pd.read_csv(PROJECT_ROOT / "project_data" / "donors_train.csv", index_col=0)

 ## <h3 style="border-bottom: 4px solid RED; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">2.  Applying pre processing</h3>

 ### <h4 style="border-bottom: 4px solid RED; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">2.1 Data inconsistencies</h4>

For now, treating them all as missing, but they are clearly MNAR

In [19]:
rules = {
    "DONOR_AGE": lambda s: s <  0,
    "CHILDREN": lambda s: s  <  0,
    "TARGET_B": lambda s: ~s.isin([0, 1]),
    "RECENT_RESPONSE_PROP": lambda s: (s < 0) | (s > 1),
    "RECENT_CARD_RESPONSE_PROP": lambda s: (s < 0) | (s > 1),
    "RECENT_AVG_CARD_GIFT_AMT": lambda s: s < 0 # new
}

def force_incoherence_to_null(df, rules):
    for col, condition in rules.items():
        if col not in df.columns:
            continue
        mask = condition(df[col])
        mask = mask.fillna(False)
        df.loc[mask, col] = np.nan
    return df

force_incoherence_to_null(data, rules)

,CARD_PROM_12,CHILDREN,DONOR_AGE,DONOR_GENDER,FILE_CARD_GIFT,FREQUENCY_STATUS_97NK,HOME_OWNER,INCOME_GROUP,LAST_GIFT_AMT,LIFETIME_CARD_PROM,...,RECENT_AVG_GIFT_AMT,RECENT_CARD_RESPONSE_COUNT,RECENT_CARD_RESPONSE_PROP,RECENT_RESPONSE_COUNT,RECENT_RESPONSE_PROP,RECENT_STAR_STATUS,SES,URBANICITY,WEALTH_RATING,TARGET_B
CONTROL_NUMBER,,,,,,,,,,,,,,,,,,,,,
61745,4.0,3.0,33.0,M,0.00000,1.0,H,5.0,20.0000,9.0,...,17.50,NaN,0.000,2.0,0.154,0.0,2,T,NaN,1.0
112703,3.0,2.0,NaN,F,1.00000,1.0,U,NaN,15.0000,6.0,...,15.00,1.0,0.250,1.0,0.100,0.0,3,R,NaN,1.0
166437,4.0,2.0,NaN,F,7.00000,3.0,H,4.0,10.0000,17.0,...,10.67,0.0,0.000,3.0,0.231,1.0,1,U,NaN,0.0
170621,4.0,NaN,61.0,M,13.00000,1.0,H,6.0,11.0000,28.0,...,10.00,2.0,0.286,2.0,0.111,0.0,1,U,NaN,0.0
44428,6.0,0.0,75.0,M,3.00000,4.0,H,3.0,7.0000,9.0,...,5.40,3.0,0.600,5.0,0.500,0.0,2,C,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34797,5.0,0.0,84.0,M,6.00000,1.0,H,6.0,10.0000,32.0,...,NaN,1.0,0.100,2.0,0.095,6.0,2,C,3.0,0.0
7550,7.0,4.0,86.0,F,16.00000,3.0,H,2.0,10.0000,33.0,...,7.50,2.0,0.222,4.0,0.211,1.0,2,U,3.0,0.0
115215,6.0,3.0,NaN,M,2.00000,3.0,U,1.0,10.0000,12.0,...,9.00,2.0,0.250,4.0,0.235,0.0,2,T,NaN,0.0


 ### <h4 style="border-bottom: 4px solid RED; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">2.2 Missing Values</h4>

 ### <h4 style="border-bottom: 4px solid RED; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">2.2.1 Variables with substantial missingness </h4>

These variables should still be retained, but their missing values will need more careful treatment:
- `WEALTH_RATING`: since it has the highest proportion of missing values, we should **review it carefully during preprocessing**, possibly together with a **missing-value indicator**.
- `DONOR_AGE`: missing values are too frequent to ignore, so the variable should be **kept and imputed carefully**.
- `INCOME_GROUP`: this variable should also be **retained**, but its missingness should be handled with care.
- `MONTHS_SINCE_LAST_PROM_RESP`: since our EDA suggested that missingness may contain useful information, it may be worth creating a **missing-value flag** before imputation.
- `FREQUENCY_STATUS_97NK`: the same reasoning may apply here, since its missingness may also be informative.
- `HOME_OWNER`: missing values in this variable should not be treated as irrelevant, as the EDA suggests they may carry predictive signal.


**Wealth Rating** 

In [30]:
corr_with_wealth = (
    data.select_dtypes(include=np.number)
    .corr().abs()
    .loc[:, "WEALTH_RATING"]
    .drop("WEALTH_RATING")
    .to_frame("Absolute correlation")
    .sort_values("Absolute correlation", key=lambda s: s.abs(), ascending=False)
)

corr_with_wealth.head(5)

,Absolute correlation
MEDIAN_HOUSEHOLD_INCOME,0.517262
PER_CAPITA_INCOME,0.456485
INCOME_GROUP,0.354637
MEDIAN_HOME_VALUE,0.334830
PCT_OWNER_OCCUPIED,0.205269


In [31]:
missing_income = data['MEDIAN_HOUSEHOLD_INCOME'].isna().sum()
total_rows = len(data)
missing_pct = missing_income / total_rows * 100

print(f"Missing MEDIAN_HOUSEHOLD_INCOME: {missing_income} rows")
print(f"Missing percentage: {missing_pct:.2f}%")

Missing MEDIAN_HOUSEHOLD_INCOME: 270 rows
Missing percentage: 1.99%
